
# Clean CV Experiment Analysis

This notebook provides a streamlined analysis of the 5×5 repeated
cross‑validation experiments used for the atomic models.  Each plot cell
produces a **1×4 grid** (one panel per element: H, C, N, O) for a single
combination of plot type / property / metric / split.

**Outline:**
1. Setup and load data
2. Loss curves for all models 
3. Compute metrics for 5×5 CV
4. Pre-compute element & cluster metrics + define plot functions
5. Validation / test comparisons (box plots, Tukey CI, Tukey tables, parity)
6. Cluster label comparisons
7. Summary tables



## 1. Setup and Load Data


In [ ]:
import os
import json
import warnings
import importlib
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import f_oneway, levene
from statsmodels.stats.anova import AnovaRM
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# Load analysis utilities from result_analysis.py
sys.path.insert(0, os.getcwd())
import result_analysis as ra
importlib.reload(ra)

# Re-export metric functions so plotting helpers can call them without 'ra.'
concordance_correlation_coefficient = ra.concordance_correlation_coefficient
_compute_metrics_dict               = ra._compute_metrics_dict

# Plot style
sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['DejaVu Sans', 'Arial', 'Helvetica', 'Liberation Sans'],
    'font.size': 11, 'axes.titlesize': 11, 'axes.labelsize': 11,
    'xtick.labelsize': 10, 'ytick.labelsize': 10,
    'legend.fontsize': 10, 'figure.titlesize': 12,
})
warnings.filterwarnings('ignore')

# Experiment configuration
nb_dir       = os.getcwd()
project_root = os.path.abspath(os.path.join(nb_dir, '../../'))
EXPERIMENT_DIR = os.path.join(project_root, 'experiments', 'atomic')

#They will appear from the bottom up
MODEL_NAMES = ['EGNN','SG-8-5','SG-8-12','SFC2','EGFC']
#MODEL_NAMES = ['SGN2']
N_REPEATS     = 5
N_INNER_FOLDS = 5
PROPERTY_NAMES = ra.ALL_PROPS
STRATIFY_QUANTILE = 0.95
ELEMENTS       = ['H', 'C', 'N', 'O']
CLUSTER_LABELS = ['H_10', 'C_11', 'N_13', 'O_10']
MODEL_COLORS = {
    'SG-8-12': "#27d687"
}



# Thin wrappers kept for parity_overlap_1x4 compatibility

print('Configuration loaded. MODEL_NAMES:', MODEL_NAMES)


##  Compute Metrics for 5×5 CV

Define helpers that flatten the per‑molecule predictions to atom level, then
iterate over all models/folds to produce fold‑level and repeat‑averaged metrics.
(Repeat‑averaging will give 5 independent observations per method for ANOVA.)

In [ ]:


elem_metrics, cluster_metrics = ra.precompute_metrics(
    experiment_dir    = EXPERIMENT_DIR,
    models            = MODEL_NAMES,
    elements          = ELEMENTS,
    cluster_labels    = CLUSTER_LABELS,
    prop_names        = PROPERTY_NAMES,
    n_inner_folds     = N_INNER_FOLDS,
)

comp_methods = [m for m in MODEL_NAMES]
print(f'elem_metrics:    {elem_metrics.shape}')
print(f'cluster_metrics: {cluster_metrics.shape}')

In [ ]:
cluster_metrics

In [ ]:
# Plot helpers live in tmp_plot_cell_updated.py
import importlib
import tmp_plot_cell_updated as plots

importlib.reload(plots)
comp_methods = [m for m in MODEL_NAMES]

_BIND_KEYS = [
    'EXPERIMENT_DIR', 'ELEMENTS', 'CLUSTER_LABELS',
    'MODEL_COLORS', 'comp_methods', 'elem_metrics', 'cluster_metrics',
    'PROPERTY_NAMES',
    'get_folds', 'safe_read_pickle',
    'flatten_predictions_by_element', 'flatten_predictions_by_cluster',
    'rm_tukey_hsd',
]
plots.bind_notebook_globals({k: globals()[k] for k in _BIND_KEYS if k in globals()})

from tmp_plot_cell_updated import *  # keeps existing plot-call cells working

print("plot functions overridden from tmp_plot_cell_updated.py")

In [ ]:
plot_family('tukey_combined', metric='CCC', split='clusters', properties=['LI','N','|Mu|','|Q|'], save_dir = 'tukey_augmented/')
plot_family('tukey_combined', metric='CCC', split='test', properties=['LI','N','|Mu|','|Q|'], save_dir = 'tukey_augmented/')

The dashed vertical lines now bracket the best model's mean by ± the Tukey minimum significant difference q(α, k, df_err)·√(MSE/n) — any model whose mean falls inside that band is exactly the set coloured grey.

In [ ]:
plot_family('parity', models=['SGN2'], split='clusters',
            properties=['N','LI','|Mu|','|Q|'], cmap = 'winter',
            save_dir= 'parity', marginal_log = True, panel_metric = 'R2')

In [ ]:
avg_over_atoms = ra.tukey_pvalue_matrix(cluster_metrics, models=['EGNN','SG-8-5','SG-8-12','SFC2','EGFC'],
                              metric='CCC', panels=CLUSTER_LABELS, avg_over_panels = True,
                              properties=['N','LI','|Mu|','|Q|'])

In [ ]:
avg_over_both = ra.tukey_pvalue_matrix(cluster_metrics, models=['EGNN','SG-8-5','SG-8-12','SFC2','EGFC'],
                              metric='CCC', panels=CLUSTER_LABELS, avg_over_panels = True,
                              avg_over_props = True,
                              properties=['N','LI','|Mu|','|Q|'])

In [ ]:

ra.latex_metric_table(
    cluster_metrics, models=['EGNN','SG-8-5','SG-8-12','SFC2','EGFC'],
    panels=CLUSTER_LABELS, avg_over_panels=True,
    avg_over_props = True,
    orient = 'wide', landscape = False
)

In [ ]:

ra.latex_metric_table(
    cluster_metrics, models=['EGNN','SG-8-5','SG-8-12','SFC2','EGFC'],
    panels=CLUSTER_LABELS, avg_over_panels=False,
    avg_over_props = False,
    orient = 'tall', landscape = True
)


In [ ]:

DIAG_METRIC = 'CCC'  
DIAG_PROPS  = ra.DEFAULT_SUMMARY_PROPS  # ['N', 'LI', '|Mu|', '|Q|']

diag = ra.cv_anova_diagnostics(
    elem_metrics    = elem_metrics,
    cluster_metrics = cluster_metrics,
    models          = ['EGNN','SG-8-5','SG-8-12','SFC2','EGFC'],
    elements        = ELEMENTS,
    cluster_labels  = CLUSTER_LABELS,
    properties      = DIAG_PROPS,
    metric          = DIAG_METRIC,
    n_inner_folds   = N_INNER_FOLDS,
    experiment_dir  = EXPERIMENT_DIR,
)
print('Diagnostic tables computed:')
for k, v in diag.items():
    print(f'  {k}: {v.shape}')

In [ ]:
_ = ra.latex_anova_diagnostics_table(
    diag,
    models=['EGNN','SG-8-5','SG-8-12','SFC2','EGFC'],
    metric='CCC',
    panels=CLUSTER_LABELS,
    properties=DIAG_PROPS,
    panel_type='cluster',
    avg_over_panels=True,
    avg_over_props=True,
    orient = 'wide',
    show_pvalues=False,
    include_intervals=True,
    landscape=False,
)

In [ ]:

_ = ra.latex_anova_diagnostics_table(
    diag,
    models=['EGNN','SG-8-5','SG-8-12','SFC2','EGFC'],
    metric=['CCC'],
    panels=CLUSTER_LABELS,
    properties=DIAG_PROPS,
    panel_type='cluster',
    avg_over_panels=False,
    avg_over_props=False,
    orient = 'tall',
    show_pvalues=False,
    include_intervals=True,
    landscape=True,
)